## Status (created 2026-07-30) -- READ BEFORE RUNNING

**Standalone BBKNN baseline notebook.** Deliberately a SEPARATE notebook from
`batch_correct_then_cluster_baselines.ipynb` (Harmony) and `combat_then_cluster_baselines.ipynb`
(ComBat) -- same underlying helper module
(`interpretable_ssl/evaluation/batch_correct_baselines.py`) and same shared results
section (`interpretable_ssl/evaluation/rebuttal_report.py`), just a different
`CORRECTION_METHODS` value (`['bbknn']` here).

BBKNN support already existed in `batch_correct_baselines.py` (`get_bbknn_graph`, the
`'bbknn'` branch in `run_correction_method`) from the original 4-method design, but was
excluded from every notebook's `CORRECTION_METHODS` due to a suspected stale numpy/scipy
pin inside the `bbknn` PyPI package itself conflicting with the numpy/scipy pins scVI/
scArches need. This notebook installs `bbknn` with `--no-deps` (see the install cell)
so its own requirements file never gets a chance to downgrade numpy/scipy -- if that
turns out not to be enough, the import-check cell below will fail loudly and name
exactly which package is missing.

Results land in the exact same on-disk run folders/metrics.json format the other two
notebooks use (`seacell_X_bbknn`, `leiden_X_bbknn_K{n}`), so the comparison tables below
can pull in Harmony/ComBat results too, if those have already been run -- no need to
re-run anything from the other notebooks to see them side by side here.

If you already have this notebook open in another Colab tab, close it (or File >
Revert) first -- Colab autosaving from a stale tab can silently overwrite this file.


# Rebuttal experiment: BBKNN batch-correction-then-cluster baseline -- BBKNN graph -> {SEACells, Leiden}

**Purpose.** Reviewer F5RB (Question 1): *"Please compare scProto with stronger
two-step baselines, such as scPoli latent + SEACells, scVI latent + SEACells, Harmony
latent + SEACells, and BBKNN graph + SEACells."* The scPoli/Harmony/scVI arms are
already covered by the Harmony notebook; this notebook runs the BBKNN arm.

**Why BBKNN is a structurally different comparison than Harmony/ComBat/scVI:** those
three all correct expression or a PCA embedding FIRST, then a separate step (RBF kernel
+ SEACells/Leiden) builds a graph on top of the corrected result. BBKNN instead builds
its batch-balanced kNN graph directly -- for each cell, it finds its k nearest
neighbors WITHIN EACH BATCH SEPARATELY (not a global neighbor search), then merges those
per-batch neighbor sets into one graph. There is no intermediate corrected embedding at
all: the graph IS the batch correction. That graph is fed directly to SEACells/Leiden
below -- no separate RBF-on-embedding construction step, unlike every other method in
these notebooks.

This also means the "embedding-only rare-cell affinity purity" diagnostic (used in the
Harmony/ComBat notebooks to check whether a correction method already destroys rare-type
separability before any clustering algorithm runs) does not apply to BBKNN -- there is
no embedding to build that diagnostic's own ARBF graph on top of. The rare-cell
coverage/homogeneity/F1 table further down (computed on BBKNN's own graph directly) is
the fair comparison for this method instead.


## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q scarches faiss-cpu scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"
# bbknn --no-deps: installed LAST, after the numpy/scipy pins above are already in
# place -- bbknn's own requirements file is suspected to carry a stale numpy/scipy
# pin (see the Status cell), and --no-deps stops pip from acting on it and
# downgrading what scVI/scArches need. bbknn's actual runtime code does not appear to
# need anything beyond what's already installed above (umap-learn, scikit-learn) --
# if that turns out wrong, the import-check cell below will fail loudly and name the
# missing package explicitly.
!pip install bbknn --no-deps -q
# annoy: bbknn/matrix.py does `from annoy import AnnoyIndex` unconditionally at
# import time -- a real (if small) dependency, not something --no-deps could skip
# safely. Confirmed missing via ModuleNotFoundError on first run; installed
# separately here since it has no numpy/scipy pin of its own to worry about.
!pip install annoy -q


In [ ]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.
#
# Note: scarches/scvi-tools/harmonypy are installed above even though THIS notebook
# only runs BBKNN -- run_all_baselines_for_dataset always calls get_stage1_latent()
# first to build `ad` from the existing scPoli Stage-1 checkpoint, and that path needs
# scarches/scvi importable regardless of which correction method is requested
# afterward.


In [ ]:
# Ground truth for "did the install cell above actually work" -- pip's own log is
# noisy (resolver backtracking prints "Getting requirements to build wheel" errors
# for discarded candidate versions even on a fully successful install), so eyeballing
# it is unreliable. Actually importing every package we just installed is the real
# test.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'harmonypy': 'harmonypy',
    'faiss-cpu': 'faiss', 'bbknn': 'bbknn', 'annoy': 'annoy',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', '?')
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {ver})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above and check its full error "
          f"output before proceeding.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")


In [1]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py


nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [2]:
# Extra imports not already covered by nb_setup.py (which already pulls in
# run_mc_task, eval_seacell_task1/2/3, load_task1_multi, show_table, clean_run_names,
# rare_celltype_purity_table, TASK1_METRICS, TASK2_METRICS, extract_model_key via
# `from interpretable_ssl.evaluation.paper_figures import *` and
# `from interpretable_ssl.evaluation.metric_helpers.result_tables import *`).
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir

print("extra imports ready")


extra imports ready


## Config

Same hyperparameters `train_scproto.ipynb` used to produce the existing Stage-1
checkpoints for these three datasets (`cvae_epochs=50`, `batch_size=1024`) -- must
match, or `load_pretrain_checkpoint()` looks in the wrong folder. `K` (n_SEACells /
target Leiden cluster count) defaults to each dataset's `num_prototypes` from
`DATASETS`, matching `results.tex`: *"All baselines are configured to produce the same
number of metacells K as scProto."*


In [3]:
RNA_SEQ_DATASETS = ['pancreas', 'lung', 'pbmc-immune']

SKIP_IF_EXISTS = True  # skip a baseline entirely (embedding + SEACells + Leiden) if its
                        # metrics.json already exists on disk -- re-running the dataset
                        # cell after a crash/interrupt never redoes already-saved work.

CORRECTION_METHODS = ['bbknn']  # this notebook: BBKNN only (see intro cell)
METHOD_DISPLAY_NAMES = {
    'bbknn': 'BBKNN',
}

RUN_SEACELLS = True  # both SEACells AND Leiden on BBKNN's own graph -- reviewer F5RB's
                      # Q1 explicitly named "BBKNN graph + SEACells" as a requested arm.
RUN_LEIDEN = True


## Helper functions

Shared with the Harmony and ComBat notebooks -- lives in
`interpretable_ssl/evaluation/batch_correct_baselines.py` (imported below), not
redefined here. BBKNN support (`get_bbknn_graph`, and the `'bbknn'` branch in
`run_correction_method`) already existed in that module from the original 4-method
design -- this notebook just turns it back on via `CORRECTION_METHODS`, it required no
new helper-module code.


In [4]:
from interpretable_ssl.evaluation.batch_correct_baselines import run_all_baselines_for_dataset

print("baseline helper functions ready (interpretable_ssl.evaluation.batch_correct_baselines)")


baseline helper functions ready (interpretable_ssl.evaluation.batch_correct_baselines)


## Run: Pancreas

In [5]:
pancreas_results = run_all_baselines_for_dataset(
    'pancreas', correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
    run_seacells=RUN_SEACELLS, run_leiden=RUN_LEIDEN,
)


loading pancreas data
✅ Already subsetted to HVGs (4000 genes).


 captum (see https://github.com/pytorch/captum).


dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pancreas/pretrain/pretrain_ds-pancreas_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pancreas', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'tech'}
📊 EdgeDataset: 1155146 edges
   Weight range: [0.0320, 0.9688]
   umap_steps_per_epoch=500 → 512000 edges/epoch (of 1155146 total

  0%|          | 0/16 [00:00<?, ?it/s]


=== [pancreas] batch-correction method: bbknn ===
[pancreas] bbknn_d8: cached graph to /content/drive/MyDrive/models/pancreas/_cache_X_bbknn_d8_aff.npz
[waypoint init] N=16382  k=220  n_eigs=10  nnz=535440  nnz/row=32.7
[waypoint init] computing diffusion map ...
[waypoint init] diffusion map done — 10 eigenvectors


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/16382 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/16382 [00:00<?, ?it/s]

Constructing CSR matrix...
[pancreas] scProto (d=8): 0.454 +/- 0.184 (n=8 batches)
[pancreas] affinity purity saved to /content/drive/MyDrive/models/pancreas/rare_affinity_purity_pancreas.json


## Run: Lung

In [6]:
lung_results = run_all_baselines_for_dataset(
    'lung', correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
    run_seacells=RUN_SEACELLS, run_leiden=RUN_LEIDEN,
)


loading lung data
✅ Already subsetted to HVGs (4000 genes).
dataset is None, loading lung
loading lung data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [16]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.255/24.467/119.300, effk_med=63.8, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/lung/pretrain/pretrain_ds-lung_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'lung', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'batch'}
📊 EdgeDataset: 2447924 edges
   Weight range: [0.0137, 0.9491]
   umap_steps_per_epoch=500 →

  0%|          | 0/32 [00:00<?, ?it/s]


=== [lung] batch-correction method: bbknn ===
[lung] bbknn_d8: cached graph to /content/drive/MyDrive/models/lung/_cache_X_bbknn_d8_aff.npz
[waypoint init] N=32472  k=300  n_eigs=10  nnz=1474964  nnz/row=45.4
[waypoint init] computing diffusion map ...
[waypoint init] diffusion map done — 10 eigenvectors


waypoint MaxMin: 100%|██████████| 299/299 [00:00<00:00, 1085.84archetype/s]


[waypoint init] selected 300 archetype seed cells
[SEACells backend] No GPU → use_sparse=True (sparse CPU, avoids dense K)
[SEACells backend] installed SEACells has no `use_sparse` param — dropping it
Welcome to SEACells!
Using provided list of initial archetypes
Randomly initialized A matrix.
Setting convergence threshold at 0.41957
Starting iteration 1.
Completed iteration 1.
Converged after 7 iterations.
Converged after 8 iterations.
Converged after 9 iterations.
Starting iteration 10.
Completed iteration 10.
Converged after 10 iterations.


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/32472 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/32472 [00:00<?, ?it/s]

Constructing CSR matrix...
[lung] scProto (d=8): 0.586 +/- 0.208 (n=15 batches)
[lung] affinity purity saved to /content/drive/MyDrive/models/lung/rare_affinity_purity_lung.json


## Run: PBMC (Immune)

In [7]:
immune_results = run_all_baselines_for_dataset(
    'pbmc-immune', correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
    run_seacells=RUN_SEACELLS, run_leiden=RUN_LEIDEN,
)


loading pbmc-immune data
✅ Already subsetted to HVGs (4000 genes).
dataset is None, loading pbmc-immune
loading pbmc-immune data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [5]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=1.667/25.923/299.545, effk_med=62.9, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pbmc-immune/pretrain/pretrain_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pbmc-immune', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'study'}
📊 EdgeDataset: 2590828 edges
   Weight range: [0.0050, 0.9224]
   umap_ste

  0%|          | 0/33 [00:00<?, ?it/s]


=== [pbmc-immune] batch-correction method: bbknn ===
[pbmc-immune] bbknn_d8: cached graph to /content/drive/MyDrive/models/pbmc-immune/_cache_X_bbknn_d8_aff.npz
[waypoint init] N=33506  k=300  n_eigs=10  nnz=355552  nnz/row=10.6
[waypoint init] computing diffusion map ...
[waypoint init] diffusion map done — 10 eigenvectors


waypoint MaxMin: 100%|██████████| 299/299 [00:00<00:00, 1215.18archetype/s]


[waypoint init] selected 300 archetype seed cells
[SEACells backend] No GPU → use_sparse=True (sparse CPU, avoids dense K)
[SEACells backend] installed SEACells has no `use_sparse` param — dropping it
Welcome to SEACells!
Using provided list of initial archetypes
Randomly initialized A matrix.
Setting convergence threshold at 0.36422
Starting iteration 1.
Completed iteration 1.
Converged after 9 iterations.
Starting iteration 10.
Completed iteration 10.
Converged after 10 iterations.


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/33506 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/33506 [00:00<?, ?it/s]

Constructing CSR matrix...
[pbmc-immune] scProto (d=8): 0.849 +/- 0.085 (n=5 batches)
[pbmc-immune] affinity purity saved to /content/drive/MyDrive/models/pbmc-immune/rare_affinity_purity_pbmc-immune.json


## Results comparison

Pulls in **every** existing run under `MODEL_DIR/{ds}/*` (scProto, SEACells(PCA), plus
whatever's already been computed by the Harmony/ComBat notebooks, alongside the new
BBKNN baselines here) -- `load_task1_multi` / `rare_celltype_purity_table` just scan
folders + `metrics.json` / `umap_cells.csv`, they don't need to know which notebook
produced them.

Harmony is included in the comparison tables below (via `harmony_read_only_keywords`)
purely as a READ of whatever's already on disk from the Harmony notebook -- it is never
(re)computed here. If that notebook hasn't been run yet, those rows are simply absent,
not an error. (ComBat is not pulled in automatically here -- add
`{"seacell_X_combat": "SEACells (ComBat)", "leiden_X_combat": "Leiden (ComBat)"}` to
`extra_read_only` in the next cell if you want it in this notebook's tables too.)

**Reminder:** the modularity column is scored against the arbf-on-PCA graph that only
scProto is trained to match -- treat it as a plausibility check, not the primary
evidence. Purity, batch entropy, and especially the rare-cell coverage/homogeneity
table (further down) are the fair, post-hoc comparison.


In [8]:
dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

# Shared across the Harmony/ComBat/BBKNN notebooks -- see
# interpretable_ssl/evaluation/rebuttal_report.py for why this used to be ~10
# near-duplicated cells per notebook (Table 1/2, rare-cell table, 4 significance
# tests) and now is not.
from interpretable_ssl.evaluation.rebuttal_report import (
    build_model_keywords, harmony_read_only_keywords, render_full_comparison_report,
    render_realized_k_check,
)

# Harmony is included READ-ONLY here -- pulled in for side-by-side comparison if the
# Harmony notebook has already produced it. Never (re)computed by this notebook. If
# that notebook hasn't been run yet, these keys simply match nothing on disk yet, not
# an error.
MODEL_KEYWORDS = build_model_keywords(
    CORRECTION_METHODS, METHOD_DISPLAY_NAMES,
    extra_read_only=harmony_read_only_keywords(harmony_dim=8),
)
MODEL_KEYWORDS


{'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp': 'scProto',
 'seacell': 'SEACells (PCA)',
 'seacell_X_bbknn_d8': 'SEACells (BBKNN)',
 'leiden_X_bbknn_d8': 'Leiden (BBKNN)',
 'seacell_X_harmony_d8': 'SEACells (Harmony)',
 'leiden_X_harmony_d8': 'Leiden (Harmony)'}

In [9]:
report = render_full_comparison_report(
    RNA_SEQ_DATASETS, dataset_display_names, MODEL_KEYWORDS, ref_name='scProto',
)
# report also holds the underlying DataFrames (report['task1'], report['rare'],
# report['rare_sig_paired'], etc.) if you need them beyond what's printed/displayed above.


=== Table 1: community structure / batch integration ===



=== Table 2: metacell representation quality ===


  [scProto|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] run dir resolved (0.0s)
  [SEACells (BBKNN)|pancreas] resolving run dir ...
  [Leiden (BBKNN)|pancreas] resolving run dir ...
  [SEACells (Harmony)|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] reading umap_cells.csv ...  [Leiden (Harmony)|pancreas] resolving run dir ...

  [scProto|lung] resolving run dir ...
  [SEACells (PCA)|lung] resolving run dir ...
  [SEACells (PCA)|lung] run dir resolved (0.0s)
  [SEACells (PCA)|lung] reading umap_cells.csv ...
  [scProto|lung] run dir resolved (0.1s)
  [scProto|lung] reading umap_cells.csv ...
  [SEACells (Harmony)|pancreas] run dir resolved (0.2s)
  [SEACells (BBKNN)|pancreas] run dir resolved (0.2s)
  [Leiden (BBKNN)|pancreas] run dir resolved (0.2s)
  [SEACells (Harmony)|pancreas] reading umap_cells.csv ...
  [scProto|pancreas] run dir resolved (0.2s)
  [SEACells (BBKNN)|pancreas] reading umap_cells.csv ...



=== UNPAIRED rare-cell significance (Mann-Whitney U) ===


,dataset,metric,method,k,n,median,mean,std,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.437680,0.535403,0.207961,NaN,NaN,NaN
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.372628,0.370589,0.250869,0.032643,0.163214,ns
2,Pancreas,batch_rare_f1_macro,SEACells (BBKNN),220,8,0.155239,0.177425,0.218077,0.003616,0.018081,*
3,Pancreas,batch_rare_f1_macro,Leiden (BBKNN),220,8,0.133314,0.100205,0.083917,0.000453,0.002267,**
4,Pancreas,batch_rare_f1_macro,SEACells (Harmony),220,8,0.295521,0.316501,0.159421,0.014064,0.070319,ns
5,Pancreas,batch_rare_f1_macro,Leiden (Harmony),220,8,0.133361,0.223580,0.171156,0.002331,0.011655,*
6,Pancreas,batch_rare_homogeneity,scProto,219,8,0.522129,0.564854,0.155263,NaN,NaN,NaN
7,Pancreas,batch_rare_homogeneity,SEACells (PCA),220,8,0.369265,0.420592,0.195138,0.010334,0.051671,ns
8,Pancreas,batch_rare_homogeneity,SEACells (BBKNN),220,8,0.190313,0.198640,0.070060,0.000078,0.000389,***
9,Pancreas,batch_rare_homogeneity,Leiden (BBKNN),220,8,0.156982,0.164399,0.059724,0.000078,0.000389,***


...

dataset,Immune,Lung,Pancreas
method,,,
Leiden (BBKNN),"0.856 (K=300, n=300, wins=?/300) *** p_adj=1...","0.833 (K=300, n=300, wins=?/300) *** p_adj=1...","0.999 (K=220, n=220, wins=?/220) ns p_adj=0.526"
Leiden (Harmony),"0.855 (K=300, n=300, wins=?/300) *** p_adj=2...","0.841 (K=300, n=300, wins=?/300) *** p_adj=1...","1.000 (K=220, n=220, wins=?/220) ns p_adj=1"
SEACells (BBKNN),"0.829 (K=300, n=300, wins=?/300) *** p_adj=1...","0.944 (K=300, n=300, wins=?/300) * p_adj=0.0126","0.981 (K=220, n=220, wins=?/220) ** p_adj=0...."
SEACells (Harmony),"0.883 (K=300, n=300, wins=?/300) *** p_adj=4...","0.832 (K=300, n=300, wins=?/300) *** p_adj=8...","0.967 (K=220, n=220, wins=?/220) *** p_adj=0..."
SEACells (PCA),"0.965 (K=300, n=300, wins=?/300) *** p_adj=8...","0.977 (K=300, n=300, wins=?/300) ns p_adj=1","0.993 (K=220, n=220, wins=?/220) ns p_adj=1"
scProto,"1.000 (K=294, n=294) [ref]","0.996 (K=298, n=298) [ref]","1.000 (K=219, n=219) [ref]"


=== batch_entropy_per_mc: scProto vs. each same-K baseline, one-sided Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (BBKNN),"0.143 (K=300, n=300, wins=?/300) ns p_adj=1","1.141 (K=300, n=300, wins=?/300) ns p_adj=1","0.900 (K=220, n=220, wins=?/220) ns p_adj=1"
Leiden (Harmony),"1.071 (K=300, n=300, wins=?/300) ns p_adj=1","1.350 (K=300, n=300, wins=?/300) ns p_adj=1","1.327 (K=220, n=220, wins=?/220) ns p_adj=1"
SEACells (BBKNN),"0.531 (K=300, n=300, wins=?/300) ns p_adj=1","1.162 (K=300, n=300, wins=?/300) ns p_adj=1","0.923 (K=220, n=220, wins=?/220) ns p_adj=1"
SEACells (Harmony),"0.726 (K=300, n=300, wins=?/300) ns p_adj=1","1.526 (K=300, n=300, wins=?/300) ns p_adj=1","1.339 (K=220, n=220, wins=?/220) ns p_adj=1"
SEACells (PCA),"-0.000 (K=300, n=300, wins=?/300) ns p_adj=1","0.218 (K=300, n=300, wins=?/300) ns p_adj=1","-0.000 (K=220, n=220, wins=?/220) *** p_adj=..."
scProto,"-0.000 (K=294, n=294) [ref]","0.500 (K=298, n=298) [ref]","0.214 (K=219, n=219) [ref]"



=== PAIRED Table 1 significance (modularity only, recommended for that metric) ===
=== modularity_per_batch: scProto vs. each same-K baseline, PAIRED one-sided Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (BBKNN),"0.399 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.458 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.390 (K=220, n=9, wins=9.0/9) ** p_adj=0.00977"
Leiden (Harmony),"0.250 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.403 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.614 (K=220, n=9, wins=6.0/9) ns p_adj=1"
SEACells (BBKNN),"0.251 (K=300, n=5, wins=4.0/5) ns p_adj=0.312","0.361 (K=300, n=16, wins=16.0/16) *** p_adj=...","0.398 (K=220, n=9, wins=8.0/9) * p_adj=0.0195"
SEACells (Harmony),"0.551 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.625 (K=300, n=16, wins=14.0/16) ** p_adj=0...","0.566 (K=220, n=9, wins=7.0/9) ns p_adj=0.625"
SEACells (PCA),"0.569 (K=300, n=5, wins=5.0/5) ns p_adj=0.156","0.671 (K=300, n=16, wins=9.0/16) ns p_adj=1","0.658 (K=220, n=9, wins=0.0/9) ns p_adj=1"
scProto,"0.620 (K=294, n=5) [ref]","0.669 (K=298, n=16) [ref]","0.621 (K=219, n=9) [ref]"


### Realized cluster/metacell count vs. scProto's target K

Sanity check that BBKNN's downstream SEACells/Leiden actually landed at (approximately)
scProto's own `num_prototypes` for each dataset, per the paper's own protocol ("All
baselines are configured to produce the same number of metacells K as scProto"). Reads
directly from each run's saved outputs -- no recompute.


In [10]:
target_k = {ds: DATASETS[ds]['num_prototypes'] for ds in RNA_SEQ_DATASETS}
# 'bbknn_d8' matches the dimension-qualified tag BBKNN now runs under (see
# batch_correct_baselines.DIM_MATCHED_METHODS) -- render_realized_k_check takes the
# literal on-disk suffix, it doesn't know about dimension-matching itself.
_ = render_realized_k_check(RNA_SEQ_DATASETS, DATASETS, method='bbknn_d8')


,,n_clusters,resolution,target_k,matches_target
dataset,run,,,,
pancreas,leiden_X_bbknn_d8_K220,220.0,32.0,220,True
lung,leiden_X_bbknn_d8_K300,300.0,64.0,300,True
pbmc-immune,leiden_X_bbknn_d8_K300,300.0,32.0,300,True


,,n_actual,target_k,matches_target
dataset,method,,,
pancreas,bbknn_d8,220,220,True
lung,bbknn_d8,300,300,True
pbmc-immune,bbknn_d8,300,300,True


## Embedding-only rare-cell affinity purity (no downstream clustering)

**Does not apply to BBKNN** -- this diagnostic (used in the Harmony/ComBat notebooks)
needs a corrected EMBEDDING to build its own ARBF affinity graph on top of, so it can
isolate the embedding from whichever clustering algorithm runs on top of it. BBKNN has
no embedding at all -- its batch-balanced kNN graph IS the correction, fed directly to
SEACells/Leiden. The cell below will simply show no BBKNN row (nothing to compute), not
an error -- the rare-cell coverage/homogeneity/F1 table in the Results comparison
section above (scored on BBKNN's own graph directly, post-clustering) is the fair
comparison for this method instead.


In [11]:
from interpretable_ssl.evaluation.batch_correct_baselines import load_and_compare_affinity_purity

df_affinity_purity = load_and_compare_affinity_purity(
    RNA_SEQ_DATASETS, dataset_display_names=dataset_display_names,
)
df_affinity_purity


,dataset,method,dim,n,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,Raw PCA (uncorrected),50,8,0.385,0.164,7.0,0.0742,0.0742,ns
1,Pancreas,scProto,8,8,0.454,0.184,NaN,NaN,NaN,NaN
2,Lung,Raw PCA (uncorrected),50,15,0.559,0.194,11.0,0.0240,0.0240,*
3,Lung,scProto,8,15,0.586,0.208,NaN,NaN,NaN,NaN
4,Immune,Raw PCA (uncorrected),50,5,0.827,0.050,4.0,0.3125,0.3125,ns
5,Immune,scProto,8,5,0.849,0.085,NaN,NaN,NaN,NaN
